# Equivalent noise for majority vote



In [27]:
import stim


base_circuit = stim.Circuit(
"""
# initialize data qubits and pairs
RX 0 1
R 10 11
CX 0 10 1 11
MXX 0 1
##### first
Z_ERROR(0.001) 0 1
RX 2
R 3
CX 2 3
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 3
Z_ERROR(0.001) 0 1
CX 2 0 3 1
MX(0.001) 2 3
DETECTOR(0) rec[-1] rec[-2] rec[-3]
# MEASURE PAULI ON THE DATA QUBITS
MXX 0 10
DETECTOR(1, 1) rec[-1]
MXX 1 11
DETECTOR(2, 1) rec[-1]
MZZ 0 10
DETECTOR(3, 1) rec[-1]
MZZ 1 11
DETECTOR(4, 1) rec[-1]
# OBSERVABLE_INCLUDE(0) X0 X10
# OBSERVABLE_INCLUDE(1) Z0 Z10
# OBSERVABLE_INCLUDE(2) X1
# OBSERVABLE_INCLUDE(3) Z1
"""
)


base_circuit.detector_error_model(allow_gauge_detectors=True)
# base_circuit.explain_detector_error_model_errors()


stim.DetectorErrorModel('''
    error(0.0119678) D0
    error(0.001998) D0 D1
    error(0.001998) D0 D2
    error(0.00050025) D0 D4
    error(0.5) D3 D4
    error(0.00050025) D4
    detector(0) D0
    detector(1, 1) D1
    detector(2, 1) D2
    detector(3, 1) D3
    detector(4, 1) D4
''')

In [28]:
# class noise_source:
#     def __init__(self, p, symplectic: List[bool], measurement: List[bool]):
#         self.p = p
#         self.symplectic = symplectic
#         self.measurement = measurement

# ##### first
# RX 2
# R 3
# CX 2 3
# PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 3
# Z_ERROR(0.001) 0
# Z_ERROR(0.001) 1
# CX 2 0 3 1
# X_ERROR(0.001) 2 
# X_ERROR(0.001) 3
# ##### second
# Z_ERROR(0.001) 0
# Z_ERROR(0.001) 1
# RX 4
# R 5
# CX 4 5
# PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 5
# Z_ERROR(0.001) 0
# Z_ERROR(0.001) 1
# CX 4 0 5 1
# X_ERROR(0.001) 4
# X_ERROR(0.001) 5
# ##### third
# Z_ERROR(0.001) 0 
# Z_ERROR(0.001) 1
# RX 6
# R 7
# CX 6 7
# PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 7
# Z_ERROR(0.001) 0 
# Z_ERROR(0.001) 1
# CX 6 0 7 1
# X_ERROR(0.001) 6
# X_ERROR(0.001) 7
# """

In [29]:
base_circuit = stim.Circuit(
"""
# initialize data qubits and pairs
RX 0 1
R 10 11
CX 0 10 1 11
MXX 0 1
##### first
Z_ERROR(0.001) 0 1
RX 2
R 3
CX 2 3
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 3
Z_ERROR(0.001) 0 1
CX 2 0 3 1
MX(0.001) 2 3
DETECTOR(0) rec[-1] rec[-2]
##### second
Z_ERROR(0.001) 0 1
RX 4
R 5
CX 4 5
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 5
Z_ERROR(0.001) 0 1
CX 4 0 5 1
MX(0.001) 4 5
DETECTOR(1) rec[-1] rec[-2]
##### third
Z_ERROR(0.001) 0 1
RX 6
R 7
CX 6 7
PAULI_CHANNEL_1(0.0005, 0.0005, 0.01) 7
Z_ERROR(0.001) 0 1
CX 6 0 7 1
MX(0.001) 6 7
DETECTOR(2) rec[-1] rec[-2]
# MEASURE PAULI ON THE DATA QUBITS
MXX 0 10
DETECTOR(1, 1) rec[-1]
MXX 1 11
DETECTOR(2, 1) rec[-1]
MZZ 0 10
DETECTOR(3, 1) rec[-1]
MZZ 1 11
DETECTOR(4, 1) rec[-1]
"""
)

base_circuit.detector_error_model(allow_gauge_detectors=True)


stim.DetectorErrorModel('''
    error(0.0119678) D0
    error(0.5) D0 D1 D2
    error(0.001998) D0 D1 D2 D3
    error(0.001998) D0 D1 D2 D4
    error(0.00050025) D0 D5
    error(0.0119678) D1
    error(0.001998) D1 D2 D3
    error(0.001998) D1 D2 D4
    error(0.00050025) D1 D5
    error(0.0119678) D2
    error(0.001998) D2 D3
    error(0.001998) D2 D4
    error(0.00050025) D2 D6
    error(0.001) D5
    error(0.5) D5 D6
    error(0.00050025) D6
    detector(0) D0
    detector(1) D1
    detector(2) D2
    detector(1, 1) D3
    detector(2, 1) D4
    detector(3, 1) D5
    detector(4, 1) D6
''')

Look at the dem above. we are going to turn in into an equivalent noise model for a repeated measurement 3 times.
The mapping rules are:

```
D0 - first measurement
D1 - second measrement
D2 - third measurement
D3 - Z0 error
D4 - Z1 error
D5 - X0 error
D6 - X1 error
```

equivalent error mechains has:
p: probability for error.
M: measurement error (at least two physical measurement failed)
Xi: X error on the i'th qubit
Zi: Z error on the i'th qubit

do:
1. Go over the dem and discard all the Gauge lines (probability 0.5 and only detectors are D3 D4 D5 D6).
2. Go over every combination of one or two error mechanichms and:
    - Xor the detectors so we will get an equivalent error mechanism.
    - probability is the multiplication of both (maybe we need to also take into account (1-p) of all the other probabilities, should we? or just assume other order won't change.)
    - write which detectors are flipped.
    - calculate equivalent error: 
        - D0, D1 and D2 are majority voted. meaning that we write M error just if at least two have flipped.
        - X0, X1, Z0, Z1 are just xored so this happens already in the previous step.
    - push the equivalent error mechanism to a list.


Some of the building blocks are implemented [here](https://github.com/shohamjac/injection-with-erasure-qubits/blob/main/Combination_searching.ipynb).


At the end, this equivalent noise model should be entered into the model specified in equivalent_bell_bias.md.

In [30]:
from itertools import combinations
import collections

# Detector layout in `base_circuit`:
#   D0, D1, D2 — three repeated syndrome measurements (majority-voted into M)
#   D3 -> Z0,  D4 -> Z1,  D5 -> X0,  D6 -> X1
SYNDROME_DETS = frozenset({0, 1, 2})
LOGICAL_MAP = {3: "Z0", 4: "Z1", 5: "X0", 6: "X1"}

# 1. Walk the DEM, keep `error` instructions, drop any with p == 0.5 (gauge).
dem = base_circuit.detector_error_model(allow_gauge_detectors=True)
raw_mechanisms = []
for instr in dem:
    if instr.type != "error":
        continue
    p = instr.args_copy()[0]
    if p == 0.5:
        continue
    dets = frozenset(
        t.val for t in instr.targets_copy() if t.is_relative_detector_id()
    )
    raw_mechanisms.append((p, dets))

# 2. Detector set -> equivalent flips on {M, X0, X1, Z0, Z1}.
def to_equivalent(dets):
    flips = set()
    if len(dets & SYNDROME_DETS) >= 2:
        flips.add("M")
    for d in dets - SYNDROME_DETS:
        flips.add(LOGICAL_MAP[d])
    return frozenset(flips)

# 3. Enumerate singles and pairs (first-order: p1 * p2 for pairs, no (1-p) factor).
equivalent_errors = []  # list[(p, frozenset[str])]
for p, dets in raw_mechanisms:
    equivalent_errors.append((p, to_equivalent(dets)))
for (p1, d1), (p2, d2) in combinations(raw_mechanisms, 2):
    equivalent_errors.append((p1 * p2, to_equivalent(d1 ^ d2)))

# Sanity report: probability summed per equivalent pattern.
agg = collections.defaultdict(float)
for p, flips in equivalent_errors:
    agg[flips] += p

print(f"raw mechanisms (post-gauge-drop): {len(raw_mechanisms)}")
print(f"equivalent_errors entries:        {len(equivalent_errors)}")
print()
for flips, p in sorted(agg.items(), key=lambda kv: -kv[1]):
    label = "I" if not flips else "+".join(sorted(flips))
    print(f"  {label:14s}  p_sum = {p:.6g}")


raw mechanisms (post-gauge-drop): 14
equivalent_errors entries:        105

  I               p_sum = 0.0359205
  M+Z0            p_sum = 0.00413947
  M+Z1            p_sum = 0.00413947
  Z0              p_sum = 0.00206973
  Z1              p_sum = 0.00206973
  X0              p_sum = 0.00204838
  X1              p_sum = 0.00102445
  M               p_sum = 0.000437916
  Z0+Z1           p_sum = 2.7944e-05
  M+X0            p_sum = 2.39475e-05
  M+X1            p_sum = 1.19738e-05
  M+X0+Z0         p_sum = 8.9935e-06
  M+X0+Z1         p_sum = 8.9935e-06
  M+Z0+Z1         p_sum = 7.98401e-06
  M+X1+Z0         p_sum = 2.9985e-06
  M+X1+Z1         p_sum = 2.9985e-06
  X1+Z0           p_sum = 2.9985e-06
  X1+Z1           p_sum = 2.9985e-06
  X0+Z0           p_sum = 2.9975e-06
  X0+Z1           p_sum = 2.9975e-06
  X0+X1           p_sum = 1.501e-06
  M+X0+X1         p_sum = 5.00501e-07


In [31]:
# Translate equivalent_errors -> stim-style E() block for `equivalent_bell_bias.md`.
#
# Mapping of flip labels to Pauli targets on (data 0, data 1, ancilla 2):
#   M  -> X2   (X on the ancilla flips its Z-basis measurement)
#   X0 -> X0,  X1 -> X1,  Z0 -> Z0,  Z1 -> Z1

FLIP_TO_PAULI = {"M": "X2", "X0": "X0", "X1": "X1", "Z0": "Z0", "Z1": "Z1"}
# Stable ordering for readability in the emitted line.
PAULI_ORDER = ["X0", "X1", "Z0", "Z1", "X2"]

# Use the aggregated probabilities from the previous cell.
agg_pruned = {flips: p for flips, p in agg.items() if flips}  # drop identity

lines = []
for flips, p in sorted(agg_pruned.items(), key=lambda kv: -kv[1]):
    paulis = sorted({FLIP_TO_PAULI[f] for f in flips}, key=PAULI_ORDER.index)
    lines.append(f"E({p:.6g}) " + " ".join(paulis))

noise_block = "\n".join(lines)
print(noise_block)


E(0.00413947) Z0 X2
E(0.00413947) Z1 X2
E(0.00206973) Z0
E(0.00206973) Z1
E(0.00204838) X0
E(0.00102445) X1
E(0.000437916) X2
E(2.7944e-05) Z0 Z1
E(2.39475e-05) X0 X2
E(1.19738e-05) X1 X2
E(8.9935e-06) X0 Z0 X2
E(8.9935e-06) X0 Z1 X2
E(7.98401e-06) Z0 Z1 X2
E(2.9985e-06) X1 Z0 X2
E(2.9985e-06) X1 Z1 X2
E(2.9985e-06) X1 Z0
E(2.9985e-06) X1 Z1
E(2.9975e-06) X0 Z0
E(2.9975e-06) X0 Z1
E(1.501e-06) X0 X1
E(5.00501e-07) X0 X1 X2
